# EDA on Retail Sales Data

**Objective:** Explore a real retail sales dataset to uncover patterns in sales trends, customer
demographics, and product category performance, and turn those patterns into concrete business
recommendations.

**Dataset:** `data/retail_sales.csv` — a real Kaggle "Retail Sales Dataset" with the columns:
`Transaction ID, Date, Customer ID, Gender, Age, Product Category, Quantity, Price per Unit, Total Amount`

**Two honest adaptations from the original task template**, since this real dataset's structure
differs slightly from a generic template:
- The dataset has only a **date** (no time-of-day), so the "day-of-week × hour" pattern becomes a
  **day-of-week** pattern instead.
- The dataset has **product category** but not individual product names, so "Top 10 best-selling
  products" becomes **Top 10 highest-spending customers** — still a top-10 ranking revealing who
  drives the most revenue, just at the customer level instead of the product level.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
np.random.seed(7)

## 1. Load Dataset & Initial Inspection

In [ ]:
DATA_PATH = "data/retail_sales.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
print("\nNull values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

**Observation:** Note any nulls or duplicate rows found above. If nulls exist in
`Age` or `Total Amount`, fill numeric gaps with the median; if a whole row is broken, drop it.
Duplicate transactions (identical rows) should be removed before analysis.

In [ ]:
# Clean: fill numeric nulls with median, drop exact duplicate rows
for col in ["Age", "Quantity", "Price per Unit", "Total Amount"]:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

df = df.drop_duplicates()
df["OrderMonth"] = df["Date"].dt.to_period("M").astype(str)
df["OrderQuarter"] = df["Date"].dt.to_period("Q").astype(str)
df["OrderDOW"] = df["Date"].dt.day_name()
print("Shape after cleaning:", df.shape)

## 2. Descriptive Statistics

In [ ]:
numeric_cols = ["Age", "Quantity", "Price per Unit", "Total Amount"]
desc = df[numeric_cols].describe().T
desc["mode"] = df[numeric_cols].mode().iloc[0]
desc[["mean", "50%", "mode", "std", "min", "max"]].rename(columns={"50%": "median"})

**Observation:** Compare `mean` vs `median` for `Total Amount` — if the mean sits
noticeably above the median, a small number of large transactions are pulling the average up,
typical of retail spend distributions.

## 3. Time Series — Monthly & Quarterly Sales Trends

In [ ]:
monthly_sales = df.groupby("OrderMonth")["Total Amount"].sum().sort_index()

plt.figure(figsize=(12, 5))
monthly_sales.plot(marker="o", color="steelblue")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=60)
plt.tight_layout()
plt.savefig("outputs/monthly_sales_trend.png", dpi=120)
plt.show()

In [ ]:
quarterly_sales = df.groupby("OrderQuarter")["Total Amount"].sum().sort_index()

plt.figure(figsize=(9, 5))
quarterly_sales.plot(kind="line", marker="o", color="seagreen")
plt.title("Quarterly Sales Trend")
plt.xlabel("Quarter")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("outputs/quarterly_sales_trend.png", dpi=120)
plt.show()

**Observation:** Note which month/quarter shows the highest total sales in your
results — that's the period to study further for seasonal causes (promotions, holidays, etc.).

## 4. Customer Demographics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins = [15, 25, 35, 45, 55, 65, 80]
labels = ["16-24", "25-34", "35-44", "45-54", "55-64", "65+"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)
age_counts = df["AgeGroup"].value_counts().sort_index()
sns.barplot(x=age_counts.index, y=age_counts.values, hue=age_counts.index, palette="crest", legend=False, ax=axes[0])
axes[0].set_title("Customer Age Group Distribution")
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("Number of Transactions")

gender_counts = df["Gender"].value_counts()
axes[1].pie(gender_counts.values, labels=gender_counts.index, autopct="%1.1f%%",
            colors=sns.color_palette("crest", len(gender_counts)))
axes[1].set_title("Gender Breakdown")

plt.tight_layout()
plt.savefig("outputs/demographics.png", dpi=120)
plt.show()

**Observation:** Note which age bracket and gender generate the most transactions —
that's the segment marketing spend and product assortment should be optimised around first.

## 5. Product Category Analysis & Top Customers

> Adapted from "Top 10 products": this dataset tracks **category**, not individual product
> names, so we rank the top 10 **customers** by spend instead — still a top-10 revenue ranking.

In [ ]:
top_customers = df.groupby("Customer ID")["Total Amount"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_customers.values, y=top_customers.index, hue=top_customers.index, palette="flare", legend=False)
plt.title("Top 10 Highest-Spending Customers")
plt.xlabel("Total Spend")
plt.ylabel("Customer ID")
plt.tight_layout()
plt.savefig("outputs/top10_customers.png", dpi=120)
plt.show()

In [ ]:
category_revenue = df.groupby("Product Category")["Total Amount"].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=category_revenue.index, y=category_revenue.values, hue=category_revenue.index, palette="flare", legend=False)
plt.title("Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.savefig("outputs/revenue_by_category.png", dpi=120)
plt.show()

**Observation:** Identify the category driving the largest share of revenue —
that's the category to prioritise for inventory and promotional budget.

## 6. Correlation Heatmap

In [ ]:
corr = df[["Age", "Quantity", "Price per Unit", "Total Amount"]].corr()

plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Matrix — Numerical Variables")
plt.tight_layout()
plt.savefig("outputs/correlation_heatmap.png", dpi=120)
plt.show()

**Observation:** `Price per Unit` and `Total Amount` typically show the strongest
positive correlation (price drives revenue more than quantity per order), while `Age` usually shows
little to no linear relationship with spend — age alone isn't a strong spend predictor here.

## 7. Extra Insight — Sales by Day of Week

> Adapted from "day-of-week × hour": this dataset has no time-of-day field, so the non-obvious
> view here is simply which **days** drive the most revenue.

In [ ]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_sales = df.groupby("OrderDOW")["Total Amount"].sum().reindex(dow_order)

plt.figure(figsize=(9, 5))
sns.barplot(x=dow_sales.index, y=dow_sales.values, hue=dow_sales.index, palette="magma", legend=False)
plt.title("Total Sales by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.savefig("outputs/sales_by_dow.png", dpi=120)
plt.show()

**Observation:** Note which day(s) stand out — if weekends outperform weekdays (or
vice versa), that's a signal for when to schedule promotions and staff accordingly.

## 8. Conclusion — Business Recommendations

Based on the analysis above (replace bracketed items with your actual findings once you run this):

1. **Double down on the peak month/quarter** identified in Section 3 — increase inventory,
   staffing, and ad spend ahead of that period rather than spreading budget evenly.
2. **Prioritise the leading product category** identified in Section 5 for homepage placement,
   bundling, and reorder priority, since it drives a disproportionate share of revenue.
3. **Target retention offers at top-spending customers** identified in Section 5 — a small group
   of customers driving outsized revenue is a strong candidate for a loyalty program.
4. **Align promotions with the strongest day(s) of the week** from Section 7 rather than running
   flat promotions every day equally.